Example 15-1. Basic example of metadata in JSON

Save this JSON as `/lakehouse/default/Files/meta/metadata.json` in the attached lakehouse before running the processing examples.

In [ ]:
[
    {
        "destination": "dbo.sales_Customers",
        "source": "Files/data/Sales/Customers",
        "format": "csv",
        "projected_columns": [
            "CustomerID",
            "FirstName",
            "LastName"
        ]
    },
    {
        "destination": "dbo.sales_Products",
        "source": "Files/data/Sales/Products",
        "format": "parquet",
        "projected_columns": []
    },
    {
        "destination": "dbo.sales_transactions",
        "source": "Files/data/Sales/Transactions",
        "format": "csv",
        "projected_columns": [],
        "custom_transform": "enrich_transactions"
    }
]

Example 15-2. The reader function

In [ ]:
def read_parquet(source):
    return spark.read.parquet(source)


def read_csv(source):
    return spark.read.options(delimiter=";", header=True).csv(source)


# Store reader functions so they can be selected by source format.
readers = {
    "parquet": read_parquet,
    "csv": read_csv,
}


def read_source(metadata_entity):
    reader = readers[metadata_entity["format"]]
    return reader(metadata_entity["source"])

Example 15-3. The transform function for processing

In [ ]:
from pyspark.sql import functions as F


def default_transform(df):
    return df.dropDuplicates()


def enrich_transactions(df):
    return df.withColumn(
        "total",
        F.col("quantity") * F.col("unit_price"),
    )


transformations = {
    "default": default_transform,
    "enrich_transactions": enrich_transactions,
}


def transform_data(df, metadata_entity):
    transform_name = metadata_entity.get("custom_transform", "default")
    return transformations[transform_name](df)

Example 15-4. The main loop, which connects all parts of the engine

For orchestration, put examples 15-2 through 15-4 in a notebook named `process_entity` with the same lakehouse attached. Mark the following defaults cell as its parameter cell so `runMultiple` can override `entity`. Run example 15-5 from a separate top notebook. These examples overwrite the destination tables.

In [ ]:
entity = None
metadata_path = "/lakehouse/default/Files/meta/metadata.json"

In [ ]:
import json

# When run standalone, read all metadata from the JSON file.
# When called from runMultiple, process only the supplied entity.
if entity is None:
    with open(metadata_path, "r") as file:
        metadata = json.load(file)
else:
    metadata = [json.loads(entity)]

# Main loop. Read data via the reader function
# select the columns needed.
# Do a transform if needed and write data.
for metadata_entity in metadata:
    df = read_source(metadata_entity)
    if metadata_entity["projected_columns"]:
        df = df.select(*metadata_entity["projected_columns"])
    df = transform_data(df, metadata_entity)
    df.write.mode("overwrite").saveAsTable(metadata_entity["destination"])

Example 15-5. The top notebook with a defined DAG calling runMultiple

In [ ]:
import json

metadata_path = "/lakehouse/default/Files/meta/metadata.json"
with open(metadata_path, "r") as file:
    metadata = json.load(file)

DAG = {
    "activities": [
        {
            "name": f"Process_{metadata_entity['destination']}".replace(".", "_"),
            "path": "process_entity",
            "args": {
                "entity": json.dumps(metadata_entity),
            },
        }
        for metadata_entity in metadata
    ]
}
notebookutils.notebook.runMultiple(DAG)